matimport os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

work_dir = r"C:\Users\HP\.gemini\antigravity\scratch\geofisika_ai"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# -------------------------------------------------------------------------
# GENERATE PLOTS & DATA FOR ALL 5 CASES
# -------------------------------------------------------------------------

# 1. Prompt 1: Anomali Bouguer
np.random.seed(42)
x = np.linspace(100, 115, 30) # Bujur Timur
y = np.linspace(-3, 3, 30)     # Lintang
X, Y = np.meshgrid(x, y)
# Simulasi anomali gravitasi: basement uplift (tinggi) di timur laut, depocenter sedimen (rendah) di barat daya
Z = 15 + 25 * np.sin(0.2 * (X - 100)) * np.cos(0.3 * (Y + 1)) - 15 * np.exp(-((X - 104)**2 + (Y + 0.5)**2)/4)

fig, ax = plt.subplots(figsize=(8, 6))
cp = ax.contourf(X, Y, Z, levels=20, cmap='Spectral_r')
cbar = fig.colorbar(cp, ax=ax)
cbar.set_label('Anomali Bouguer (mGal)', fontsize=11, fontweight='bold')
ax.contour(X, Y, Z, levels=10, colors='black', linewidths=0.6, alpha=0.7)
ax.set_title('Peta Kontur Anomali Bouguer Regional (Prompt 1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Bujur Timur (°)', fontsize=11)
ax.set_ylabel('Lintang (°)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('anomali_bouguer.png', dpi=150)
plt.close()

# 2. Prompt 2: Magnitudo & Energi Gempa
m_vals = np.array([4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0])
# Gutenberg-Richter: log10 E (Erg) = 11.8 + 1.5 M  ==> E (Joule) = 10^(4.8 + 1.5 M)
log_e_joule = 4.8 + 1.5 * m_vals
e_joule = 10 ** log_e_joule

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(m_vals, log_e_joule, 'o-', color='crimson', lw=2, markersize=8)
ax1.set_title('Hubungan Magnitudo vs Log10 Energi (Joule)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Magnitudo (M)', fontsize=11)
ax1.set_ylabel('Log10 Energi (Joule)', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.semilogy(m_vals, e_joule, 's-', color='navy', lw=2, markersize=8)
ax2.set_title('Pelepasan Energi Seismik (Skala Logaritmik)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Magnitudo (M)', fontsize=11)
ax2.set_ylabel('Energi Pelepasan (Joule)', fontsize=11)
ax2.grid(True, which='both', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('magnitudo_energi.png', dpi=150)
plt.close()

# 3. Prompt 3: Suhu vs Kelembapan Udara
hours = np.arange(72)
temp_base = 28 + 4.5 * np.sin((hours - 8) * 2 * np.pi / 24) + np.random.normal(0, 0.4, 72)
rh_base = 78 - 18 * np.sin((hours - 8) * 2 * np.pi / 24) + np.random.normal(0, 1.2, 72)
rh_base = np.clip(rh_base, 45, 98)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(hours, temp_base, color='orangered', lw=2, label='Suhu Udara (°C)')
ax1.set_ylabel('Suhu (°C)', color='orangered', fontsize=11, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.set_title('Variasi Diurnal Suhu dan Kelembapan Relatif Selama 72 Jam (Prompt 3)', fontsize=13, fontweight='bold')

ax2.plot(hours, rh_base, color='teal', lw=2, label='Kelembapan Relatif (%)')
ax2.set_ylabel('RH (%)', color='teal', fontsize=11, fontweight='bold')
ax2.set_xlabel('Waktu Pengamatan (Jam ke-)', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('suhu_kelembapan.png', dpi=150)
plt.close()

# 4. Prompt 4: Analisis DEM dan Potensi Longsor
np.random.seed(101)
elev = np.zeros((40, 40))
for i in range(40):
    for j in range(40):
        elev[i, j] = 300 + 15 * i + 8 * j + 40 * np.sin(i / 4) * np.cos(j / 5)

dy, dx = np.gradient(elev)
slope = np.arctan(np.sqrt(dx**2 + dy**2)) * (180 / np.pi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
c1 = ax1.contourf(elev, cmap='terrain', levels=20)
fig.colorbar(c1, ax=ax1, label='Elevasi DEM (mdpl)')
ax1.set_title('Model Elevasi Digital (DEM)', fontsize=12, fontweight='bold')

c2 = ax2.contourf(slope, cmap='YlOrRd', levels=[0, 7, 15, 25, 40, 65])
fig.colorbar(c2, ax=ax2, label='Kemiringan Lereng (°)')
ax2.set_title('Peta Kelas Kemiringan Lereng', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('analisis_dem_longsor.png', dpi=150)
plt.close()

# 5. Prompt 5: ENSO & SOI
months = np.arange(1, 37)
oni = 1.6 * np.sin((months - 10) * np.pi / 12) + np.random.normal(0, 0.1, 36)
soi = -1.5 * oni + np.random.normal(0, 0.3, 36)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
p1 = ax1.plot(months, oni, 'ro-', label='Oceanic Niño Index (ONI - °C)', lw=2)
p2 = ax2.plot(months, soi, 'bs--', label='Southern Oscillation Index (SOI)', lw=1.8)
ax1.axhline(0.5, color='red', linestyle=':', alpha=0.7, label='Ambang El Niño (+0.5°C)')
ax1.axhline(-0.5, color='blue', linestyle=':', alpha=0.7, label='Ambang La Niña (-0.5°C)')
ax1.set_xlabel('Bulan Pengamatan', fontsize=11)
ax1.set_ylabel('Indeks ONI (°C)', color='red', fontsize=11, fontweight='bold')
ax2.set_ylabel('Indeks SOI (Std Dev)', color='blue', fontsize=11, fontweight='bold')
ax1.set_title('Deret Waktu Indeks ENSO (ONI) dan SOI Selama 36 Bulan (Prompt 5)', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)

lines = p1 + p2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')
plt.tight_layout()
plt.savefig('enso_soi_analisis.png', dpi=150)
plt.close()

print("All plots generated successfully!")
